# 1. Linear expressions and constraints in Pyomo

Lecture: 'Linear Expressions in Pyomo' and 'Linear Constraints in Pyomo'.

*Original script: `5 Source Files/Python/pyomo_printing_demo.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *

In [ ]:
# Create a Concrete Model
model = ConcreteModel()

In [ ]:
# Define integer variables
model.x = Var(range(4), domain=NonNegativeReals)
# define a vector
cost = [10, 6, 8, 16]
# define a linear inequality constraint
model.constraint1 = Constraint(expr=sum(cost[i] * model.x[i] for i in model.x) >= 120)
# pretty print the constraint
model.constraint1.pprint()


print(10*model.x[0] + 6*model.x[1] + 8*model.x[2] + 16*model.x[3])

print()


print(sum(cost[i] * model.x[i] for i in model.x))


volume = [100, 60, 85, 160]
weight = [1.05, 2.6, 3.8, 1.6]

print(sum(cost[i] * model.x[i] for i in model.x))
print()
# Objective function
model.obj = Objective(expr=sum(cost[i] * model.x[i] for i in model.x), sense=maximize)
model.obj.pprint()

print()

In [ ]:
# Define a constraint
model.constraint1 = Constraint(expr=sum(cost[i] * model.x[i] for i in model.x) >= 32)
model.constraint1.pprint()

print()

In [ ]:
# # Define the inequality constraints
model.constraint2 = Constraint(expr=sum(weight[i] * model.x[i] for i in model.x) <= 100)
model.constraint2.pprint()

In [ ]:
# set the constraint to off
model.constraint2.deactivate()
model.constraint2.pprint()